# Single-contact spike detection (Real Data, CH19)
**The Goal** is to detect the firing of one known nerve fibre in a real rat saphenous nerve recording. This notebook only experiments on one channel, which is channel19. 
**The ground truth**: the lab marked 1,623 spikes for this unit. Here, the detections were tested against those markes. 
**The code difficulty encountered in this notebook in five ways:** the electrode picks up many fibres at once, so on a single channel  its diffecult to tell which fibre a given spike came from.
## Notebook Order


In [1]:
#imports 
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import h5py

from src.evaluate import detect_spikes, score_detections
from scipy.signal import correlate, find_peaks
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy.signal import resample
from scipy.ndimage import uniform_filter1d
import torch
import torch.nn as nn

path = r"C:\spike-denoising-synthetic\data\real"
sample_frequency = 30000 # 

# 1. Setup and data 
Load the recording and the ground truth. The data is one file per channel (float64, 30 kHz), already filtered (300-6000 Hz), and artefacts were blanked by the lab. 
ground truth is a MATLAB v7.3 file, and read by h5py

Here I worked on a 300 second window starting at stimulus 900, which is a stable stretch of the recording. All the times share one clock, and this is was verified against time_series_2.bin (starts at 1.03s, 30 kHz spacing)
### Files:
1. time_series_2.bin : the time information of the data recordings
2. 100_RhythmData_CH19_2_filtered_blanked.bin : the data recordings from channel 19
3. grouped_unit_1.mat : describes the time points of a 'good' unit which has large spikes on multiple data channels
4. 'aligned_zc_locs' which will describe when (i.e., at what time) the spike was detected on that channel. These time points will match with the 'time_series_2.bin' file

In [2]:
#Load the ground truth
ground_truth = h5py.File(path + r"\grouped_unit_1.mat", "r") # Describes the time point of a "good" unit which has large spikes
print("List of field names inside the file:", [k for k in ground_truth.keys() if not k.startswith("#")])
zc_ch19 = ground_truth["CH19"]["aligned_zc_locs"][:].flatten() # aligned zero-crossing locations

print("Spikes on CH19:", len(zc_ch19)) # ground truth spikes
print("time range (s): %.1f to %.1f" % (zc_ch19.min(), zc_ch19.max())) # 

List of field names inside the file: ['CH10', 'CH14', 'CH19', 'CH25', 'CH3', 'CH30', 'CH9', 'filename', 'group_number', 'primary_channel']
Spikes on CH19: 1623
time range (s): 1754.5 to 8265.2


In [3]:
# Define the window
ttls = np.fromfile(path + r"\TTLs_2.bin", dtype='<f8') # load stimulus pulse times from the file 

print("number of TTLs:", len(ttls)) # totsal stimulus pulses in the whole recording
print("spacing (s):", np.diff(ttls[:5])) # The gap between the first five pulses

# MATLAB counts from 1, PYTHON from 0
print("ttls[792]:", ttls[792])  
print("ttls[900]:", ttls[900]) 

window_start = ttls[900]   # the time of pulse index 900, so the 300-second window begins at the 900th stimulus pulse 
window_end   = window_start + 300 # the window ends 300 seconds after it starts

print("window: %.2f to %.2f s" % (window_start, window_end)) # print window starta nd end time
print("stimuli in window:", ((ttls >= window_start) & (ttls <= window_end)).sum()) # count how many stimulus pulses in side the 300 second window

number of TTLs: 2310
spacing (s): [4.0007     4.0007     4.00073333 4.0007    ]
ttls[792]: 3170.5945666666667
ttls[900]: 3602.6708
window: 3602.67 to 3902.67 s
stimuli in window: 75


## The loaded window:
9,000,000 samples (300 s), noise floor MAD 4.92, 71 ground truth spikes inside the window. These are what every method below is scored against

In [4]:
# Load CH19 for the 300-second window
ch19_file = path + r"\100_RhythmData_CH19_2_filtered_blanked.bin" # prefiltered and artefact blanked
start_idx = int(window_start * sample_frequency) # Convert window start time (3602s) into sample index (mapping the timestamp to a position)

n_samples = int(300 * sample_frequency) # How many samples in 300 seconds window (300 x 30000 (sample_frequency) = 9,000,000 samples). 
ch19 = np.fromfile(ch19_file, dtype='<f8', count=n_samples, offset=start_idx * 8) # load Ch19 on that window

window_end = window_start + len(ch19) / sample_frequency      
noise_mad = np.median(np.abs(ch19)) / 0.6745  # MAD is the noise estimate for this window
ground_truth_in_window = zc_ch19[(zc_ch19 >= window_start) & (zc_ch19 <= window_end)] # Ground truth spikes between the start and the end of the window

print("samples loaded:", len(ch19)) # 9,000,000
print("seconds: %.1f" % (len(ch19) / sample_frequency)) # 300
print("noise (MAD): %.2f" % noise_mad) # noise floor 
print("ground-truth spikes in window:", len(ground_truth_in_window)) # the ground truth spikes in the window


samples loaded: 9000000
seconds: 300.0
noise (MAD): 4.94
ground-truth spikes in window: 71


# 2. Characterise the spikes
This section measures how deep the ground truth spikes are. 
To look at the depth of the spike:
To hold a full spike, a 3 ms clip is chosen; 3 ms at 30 kHz is 90 samples.  
So, **half 45** is 45 samples = 1.5 ms on each side (and the full 90-sample window = 3 ms), and the spike is in the centre. 
In Extracellular recordings that measure extracellular potentials, when an axon fires an action potential, positive sodium ions rush into the cell at the active region. This leaves the extracellular space near the electrode locally negative, which the electrode records as a downward deflection. So, because of the negative-going spike amplitudes: **(5th Percentile)** is the lowest numerical value, meaning it is the largest negative drop. Physically, representing deepest, the highest-amplitude, strongest spikes (furthest away from 0).**(95th Percentile)**: This is closer to zero and represents the shallowest, smallest spikes that sit right on the edge of the background noise floor.

Once the absolute value is taken in the code (np.abs(depths)), these mathematical positions flip, on the transformed positive scale, where the 5th percentile becomes the shallow spikes and the 95th percentile becomes the deep spikes

In [7]:
# Characterise the spikes: for each ground truth spike time in the window, convert to a sample index
half = 45  # Samples to take each side of each spike. 45 samples = 1.5ms at 30 kHz. 

depths = [] # empty list to collect each spike's depth
for spike_time in ground_truth_in_window:
    spike_index = int((spike_time - window_start) * sample_frequency)  # Convert this spike's time to a sample index within the loaded window 
                                                    # Find where a spike sits inside ch19, subtract the window's start time, then multiply by the frequency.  
    
    seg = ch19[spike_index - half : spike_index + half] # Cut a 90 sample window out of Ch19 centred on the spike. 
    if len(seg) == 2 * half:  # Only keep full length segments, so skip spikes too close to the window edge to cut a full 90 samples
        depths.append(seg.min()) 
depths = np.array(depths) # The trough of each spike
ratios = np.abs(depths) / noise_mad # each depth divided by the MAD noise floor, so how many times the noise level each spike goes. 

print("spikes measured:", len(depths)) # How many spikes got meausred 
print("median depth: %.2f (%.2f x noise)" % (np.median(depths), np.median(ratios))) # median depth and median depth relative to the noise
print("5th percentile:  %.2f x noise" % np.percentile(ratios, 5)) #shallow spikes 
print("95th percentile: %.2f x noise" % np.percentile(ratios, 95)) # deepest spikes

spikes measured: 71
median depth: -11.32 (2.29 x noise)
5th percentile:  1.45 x noise
95th percentile: 3.51 x noise
